# 00 — Colab Setup

Run this notebook **first** on every new Google Colab session before Bronze/Silver/Gold notebooks.


## Why Google Drive persistence?

- **`/content` is temporary** — runtime disconnects or session expiry can delete cloned repo data and local outputs.
- **Google Drive persistence** protects long Bronze ingestion and Silver cleaning outputs.
- **For official MBA evidence**, Drive-backed outputs are **recommended**.
- **Code** stays in the GitHub repo under `/content/openf1-big-data-pipeline`.
- **Generated** `data/`, `reports/`, and `artifacts/` can live on Drive when `USE_GOOGLE_DRIVE=True`.

Set `OPENF1_DATA_ROOT` **before** importing `openf1_pipeline.config` so paths resolve to Drive.


## 1. Clone repository and install dependencies


In [3]:
# Uncomment if the repo is not already cloned in this Colab session:
!git clone https://github.com/dk546/openf1-big-data-pipeline.git
%cd /content/openf1-big-data-pipeline

Cloning into 'openf1-big-data-pipeline'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 105 (delta 23), reused 105 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 70.21 KiB | 2.51 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/openf1-big-data-pipeline


In [4]:
%pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.3 MB/s eta 0:00:00


## 2. Google Drive persistence (recommended)

When `USE_GOOGLE_DRIVE=True`:

- Mounts Google Drive
- Sets `OPENF1_DATA_ROOT=/content/drive/MyDrive/openf1_big_data_pipeline`
- Creates output folders on Drive

When `USE_GOOGLE_DRIVE=False`, outputs are written under the repo root (ephemeral `/content`).


In [5]:
import os
import sys
from pathlib import Path

USE_GOOGLE_DRIVE = True

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = Path("/content/openf1-big-data-pipeline")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")
    DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_PROJECT_ROOT)
    print(f"OPENF1_DATA_ROOT is set to: {os.environ['OPENF1_DATA_ROOT']}")
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT is not set — using repo-local outputs.")

print("Code / PROJECT_ROOT (repo):", PROJECT_ROOT)

Mounted at /content/drive
OPENF1_DATA_ROOT is set to: /content/drive/MyDrive/openf1_big_data_pipeline
Code / PROJECT_ROOT (repo): /content/openf1-big-data-pipeline


## 3. Create output directories and verify paths

Import `config` only **after** `OPENF1_DATA_ROOT` is set (when using Drive).


In [6]:
from openf1_pipeline.config import ensure_project_directories, get_project_root

paths = ensure_project_directories()

print("PROJECT_ROOT (code):", get_project_root())
for key in [
    "OUTPUT_ROOT",
    "DATA_DIR",
    "BRONZE_DIR",
    "SILVER_DIR",
    "GOLD_DIR",
    "REPORTS_DIR",
    "ARTIFACTS_DIR",
    "MANIFESTS_DIR",
    "SCHEMAS_DIR",
    "PIPELINE_LOGS_DIR",
]:
    print(f"{key}: {paths[key]}")

PROJECT_ROOT (code): /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts
MANIFESTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests
SCHEMAS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/schemas
PIPELINE_LOGS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/pipeline_logs


## 4. Verify package imports


In [7]:
import pandas as pd
import requests

from openf1_pipeline.config import SEASONS, ENDPOINTS, OPENF1_BASE_URL
from openf1_pipeline.ingestion.openf1_client import OpenF1Client

print("SEASONS:", SEASONS)
print("ENDPOINTS:", len(ENDPOINTS))
print("OpenF1 base URL:", OPENF1_BASE_URL)
print("OpenF1Client OK:", OpenF1Client().base_url)

SEASONS: [2023, 2024, 2025]
ENDPOINTS: 10
OpenF1 base URL: https://api.openf1.org/v1
OpenF1Client OK: https://api.openf1.org/v1
